# Exercice 2: Classification system with KNN - To Loan or Not To Loan

## Imports

Import some useful libraries

In [79]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

## a. Getting started

### Data loading

The original dataset comes from the Kaggle's [Loan Prediction](https://www.kaggle.com/ninzaami/loan-predication) problem. The provided dataset has already undergone some processing, such as removing some columns and invalid data. Pandas is used to read the CSV file.

In [80]:
data = pd.read_csv("loandata.csv")

Display the head of the data.

In [81]:
data.head()

,Gender,Married,Education,TotalIncome,LoanAmount,CreditHistory,LoanStatus
0,Male,Yes,Graduate,6091.0,128.0,1.0,N
1,Male,Yes,Graduate,3000.0,66.0,1.0,Y
2,Male,Yes,Not Graduate,4941.0,120.0,1.0,Y
3,Male,No,Graduate,6000.0,141.0,1.0,Y
4,Male,Yes,Graduate,9613.0,267.0,1.0,Y


Data's columns:
* **Gender:** Applicant gender (Male/ Female)
* **Married:** Is the Applicant married? (Y/N)
* **Education:** Applicant Education (Graduate/ Not Graduate)
* **TotalIncome:** Applicant total income (sum of `ApplicantIncome` and `CoapplicantIncome` columns in the original dataset)
* **LoanAmount:** Loan amount in thousands
* **CreditHistory:** Credit history meets guidelines
* **LoanStatus** (Target)**:** Loan approved (Y/N)

### Data preprocessing

Define a list of categorical columns to encode.

In [82]:
categorical_columns = ["Gender", "Married", "Education", "LoanStatus"]

Encode categorical columns using the [`OrdinalEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) of scikit learn.

In [83]:
data[categorical_columns] = OrdinalEncoder().fit_transform(data[categorical_columns])

Split into `X` and `y`.

In [84]:
X = data.drop(columns="LoanStatus")
y = data.LoanStatus

Normalize data using the [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) of scikit learn.

In [85]:
X[X.columns] = StandardScaler().fit_transform(X[X.columns])

Convert `y` type to `int` 

In [86]:
y = y.astype(int)

Split dataset into train and test sets.

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Observation from the students
**We noticed that StandardScaler is fitted on the entire dataset before the train/test split.**
**Doesn't this introduce a slight data leakage, since the scaling include information from the test set?**

## b. Dummy classifier

Build a dummy classifier that takes decisions randomly.

In [88]:
class DummyClassifier():
    
    def __init__(self):
        """
        Initialize the class.
        """
        self.rng = np.random.default_rng(42) # fixed seed for reproducibility
        self.classes = None
    
    def fit(self, X, y):
        """
        Fit the dummy classifier.
        
        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_samples, n_features)
            Training data.
        y : Numpy array or Pandas DataFrame of shape (n_samples,)
            Target values.
        """
        self.classes = np.unique(y)
    
    def predict(self, X):
        """
        Predict the class labels for the provided data.

        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_queries, n_features)
            Test samples.

        Returns
        -------
        y : Numpy array or Pandas DataFrame of shape (n_queries,)
            Class labels for each data sample.
        """
        return self.rng.choice(self.classes, size=len(X))

Implement a function to evaluate the performance of a classification by computing the accuracy ($N_{correct}/N$).

In [89]:
def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

Compute the performance of the dummy classifier using the provided test set.

In [90]:
dummy = DummyClassifier()
dummy.fit(X_train, y_train)
y_predict = dummy.predict(X_test)
print("Dummy accuracy:", accuracy_score(y_test, y_predict))

Dummy accuracy: 0.5833333333333334


## c. K-Nearest Neighbors classifier

Build a K-Nearest Neighbors classifier using an Euclidian distance computation and a simple majority voting criterion.

In [91]:
class KNNClassifier():
    
    def __init__(self, n_neighbors=3):
        """
        Initialize the class.
        
        Parameters
        ----------
        n_neighbors : int, default=3
            Number of neighbors to use by default.
        """
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """
        Fit the k-nearest neighbors classifier.
        
        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_samples, n_features)
            Training data.
        y : Numpy array or Pandas DataFrame of shape (n_samples,)
            Target values.
        """
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=int)
    
    @staticmethod
    def _euclidian_distance(a, b):
        """
        Utility function to compute the euclidian distance.
        
        Parameters
        ----------
        a : Numpy array or Pandas DataFrame
            First operand.
        b : Numpy array or Pandas DataFrame
            Second operand.
        """
        diff = a - b
        return np.sqrt(np.sum(diff** 2, axis=1))
    
    def predict(self, X):
        """
        Predict the class labels for the provided data.

        Parameters
        ----------
        X : Numpy array or Pandas DataFrame of shape (n_queries, n_features)
            Test samples.

        Returns
        -------
        y : Numpy array or Pandas DataFrame of shape (n_queries,)
            Class labels for each data sample.
        """
        X = np.asarray(X, dtype=float)
        predictions = []

        for test_point in X:
            # 1. Compute distances to all training points
            distances = self._euclidian_distance(test_point, self.X_train)
            # 2. Sort distances and select the k nearest neighbors
            nearest_indices = np.argsort(distances)[:self.n_neighbors]
            # 3. Retrieve corresponding labels of the neighbors
            nearest_labels = self.y_train[nearest_indices]
            # 4. Pick the most frequent label
            most_common_label = np.bincount(nearest_labels.astype(int)).argmax()

            predictions.append(most_common_label)

        return np.array(predictions)


Compute the performance of the system as a function of $k = 1...7$.

In [92]:
results = {}
for k in range(1, 8):
    knn = KNNClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    results[k] = accuracy_score(y_test, knn.predict(X_test))
    print(f"k = {k}: accuracy = {results[k]:.4f}")

best_k = max(results, key=results.get)
print(f"Winner: k = {best_k} with accuracy = {results[best_k]:.4f}")

k = 1: accuracy = 0.6979
k = 2: accuracy = 0.6354
k = 3: accuracy = 0.7917
k = 4: accuracy = 0.7396
k = 5: accuracy = 0.8125
k = 6: accuracy = 0.7812
k = 7: accuracy = 0.8021
Winner: k = 5 with accuracy = 0.8125


Run the KNN algorithm using only the features `TotalIncome` and `CreditHistory`.

In [93]:
features = ["TotalIncome", "CreditHistory"]
knn = KNNClassifier(n_neighbors=3)
knn.fit(X_train[features], y_train)
print(f"Accuracy: {accuracy_score(y_test, knn.predict(X_test[features])):.4f}")

Accuracy: 0.7812


Re-run the KNN algorithm using the features `TotalIncome`, `CreditHistory` and `Married`.

In [94]:
features = ["TotalIncome", "CreditHistory", "Married"]
knn = KNNClassifier(n_neighbors=3)
knn.fit(X_train[features], y_train)
print(f"Accuracy: {accuracy_score(y_test, knn.predict(X_test[features])):.4f}")

Accuracy: 0.8646


Re-run the KNN algorithm using all features.

In [95]:
knn = KNNClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, knn.predict(X_test)):.4f}")

Accuracy: 0.7917
